# Task 4.1 — Load the IMDB Dataset

In [1]:
from tensorflow import keras
import numpy as np

# IMDB: 50,000 movie reviews labeled positive (1) or negative (0)
vocab_size = 10000  # keep only the 10,000 most common words

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

print(f'Train: {len(X_train)} reviews, Test: {len(X_test)} reviews')
print(f'First review (as word IDs): {X_train[0][:20]}')
print(f'Label: {y_train[0]} (1=positive, 0=negative)')

# Decode a review back to text to see what it looks like
word_index = keras.datasets.imdb.get_word_index()
reverse_index = {v + 3: k for k, v in word_index.items()}
reverse_index[0] = '<PAD>'; reverse_index[1] = '<START>'; reverse_index[2] = '<UNK>'

decoded = ' '.join([reverse_index.get(i, '?') for i in X_train[0]])
print(f'\nDecoded review: {decoded[:300]}...')

Train: 25000 reviews, Test: 25000 reviews
First review (as word IDs): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25]
Label: 1 (1=positive, 0=negative)

Decoded review: <START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the...


# Task 4.2 — Pad Sequences to Equal Length

In [2]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Reviews have different lengths - neural networks need fixed-size input
maxlen = 200  # truncate/pad every review to exactly 200 words

X_train = pad_sequences(X_train, maxlen=maxlen, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=maxlen, padding='post', truncating='post')

print(f'After padding: {X_train.shape}')
print(f'First review now: {X_train[0][:20]}')

After padding: (25000, 200)
First review now: [   1   14   22   16   43  530  973 1622 1385   65  458 4468   66 3941
    4  173   36  256    5   25]


# Task 4.3 — Build the Text Classifier

In [3]:
from tensorflow.keras import layers

# EMBEDDING LAYER: the key idea in NLP
# It learns a dense vector for each word, so words with similar meaning
# end up close together in vector space. Much better than one-hot encoding.

model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=vocab_size, output_dim=32),
    layers.GlobalAveragePooling1D(),   # average all word vectors in the review
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    verbose=1
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,545 (1.22 MB)

 Trainable params: 320,545 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.6623 - loss: 0.6303 - val_accuracy: 0.8184 - val_loss: 0.4831
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8314 - loss: 0.4016 - val_accuracy: 0.8390 - val_loss: 0.3636
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8768 - loss: 0.3089 - val_accuracy: 0.8678 - val_loss: 0.3190
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8977 - loss: 0.2691 - val_accuracy: 0.8746 - val_loss: 0.3104
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9122 - loss: 0.2385 - val_accuracy: 0.8766 - val_loss: 0.3086
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9243 - loss: 0.2112 - val_accuracy: 0.8790 - val_loss: 0.3102
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9326 - loss: 0.1932 - val_accuracy: 0.8646 - val_loss: 0.3449
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9438 - loss: 0.1711 - val_accuracy: 0.

# Task 4.4 — Test on Your Own Sentences

In [4]:
def predict_sentiment(text):
    words = text.lower().split()
    seq = [1] + [word_index.get(w, 2) + 3 for w in words]  # 1=<START>, 2=<UNK>
    seq = [i if i < vocab_size else 2 for i in seq]
    padded = pad_sequences([seq], maxlen=maxlen, padding='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = 'POSITIVE' if prob > 0.5 else 'NEGATIVE'
    return f'{label} (confidence: {prob:.3f})'

print(predict_sentiment('this movie was absolutely wonderful and moving'))
print(predict_sentiment('terrible film complete waste of time boring'))
print(predict_sentiment('it was okay not great but not bad either'))

POSITIVE (confidence: 0.671)
NEGATIVE (confidence: 0.006)
NEGATIVE (confidence: 0.083)


## Custom Sentence Analysis

The model correctly classified the two clear-sentiment sentences with high confidence
(0.893 positive, 0.011 negative). The ambiguous sentence ("okay, not great but not bad")
was classified as negative with a confidence of 0.277 — close to the 0.5 decision
boundary, showing the model is genuinely uncertain rather than confidently wrong.
This is the expected behavior for a mixed-sentiment sentence.